#  Preparação de Dados - Trabalho Estudantil
## Extração e Tratamento das Variáveis Q007 e Q008

**Notebook 2/7** - Série: Trabalho Estudantil e Desempenho no ENEM

---

##  Objetivos

1. Carregar o dataset original do ENEM 2023
2. Extrair variáveis relacionadas ao trabalho estudantil (Q007, Q008)
3. Integrar com variáveis já processadas (notas, socioeconômicas)
4. Realizar limpeza e tratamento de dados
5. Criar variáveis derivadas para análise
6. Salvar dataset processado para análises posteriores

---

## 1⃣ Setup Inicial

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

print(" Bibliotecas carregadas")

 Bibliotecas carregadas


In [26]:
# Definir caminhos
PROJECT_ROOT = Path('/home/interas/faculdade/ciencia-dados/enem-data-exploration')
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'interim' / 'unzipped_2023' / 'DADOS'
PROCESSED_DIR = DATA_DIR / 'processed'

# Arquivos
RAW_FILE = RAW_DIR / 'MICRODADOS_ENEM_2023.csv'
EXISTING_FILE = PROCESSED_DIR / 'enem_2023.parquet'  # Dataset já processado
OUTPUT_FILE = PROCESSED_DIR / 'enem_2023_trabalho_estudantil.parquet'

print(f" Dataset original: {RAW_FILE.exists()}")
print(f" Dataset processado existente: {EXISTING_FILE.exists()}")

 Dataset original: True
 Dataset processado existente: False


---

## 2⃣ Carregar Dataset Existente

Primeiro, vamos carregar o dataset já processado que contém as notas e variáveis socioeconômicas básicas:

In [27]:
%%time
# Verificar se dataset processado existe, senão usar o sample
if EXISTING_FILE.exists():
    print("Carregando dataset processado existente...")
    df_base = pd.read_parquet(EXISTING_FILE)
else:
    # Usar o sample que já foi criado anteriormente
    SAMPLE_FILE = DATA_DIR / 'interim' / 'enem_2023_sample.csv'
    if SAMPLE_FILE.exists():
        print("Dataset processado não encontrado. Carregando sample...")
        df_base = pd.read_csv(SAMPLE_FILE)
    else:
        print("Criando dataset base a partir do arquivo original...")
        # Carregar apenas colunas essenciais
        colunas_base = [
            'NU_INSCRICAO', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 
            'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'Q006', 'TP_ESCOLA', 
            'TP_SEXO', 'SG_UF_PROVA'
        ]
        df_base = pd.read_csv(RAW_FILE, sep=';', encoding='latin1', 
                              usecols=colunas_base, low_memory=False)
        
        # Criar variável de nota média
        notas_cols = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']
        df_base['NOTA_MEDIA_5'] = df_base[notas_cols].mean(axis=1)

print(f"\nDataset base carregado:")
print(f"  Linhas: {len(df_base):,}")
print(f"  Colunas: {len(df_base.columns)}")
print(f"\nColunas disponíveis:")
print(df_base.columns.tolist())

Dataset processado não encontrado. Carregando sample...

Dataset base carregado:
  Linhas: 1,000
  Colunas: 27

Colunas disponíveis:
['NU_NOTA_MT', 'NU_NOTA_LC', 'NU_NOTA_CH', 'NU_NOTA_CN', 'NU_NOTA_REDACAO', 'TP_ESCOLA', 'CO_MUNICIPIO_PROVA', 'NO_MUNICIPIO_PROVA', 'Q001', 'Q002', 'Q006', 'Q022', 'Q024', 'Q025', 'IN_TREINEIRO', 'CO_MUNICIPIO_PROVA_str', 'UF_CODE_PROVA', 'SG_UF_PROVA', 'REGIAO_ID_PROVA', 'REGIAO_NOME_PROVA', 'NOTA_MEDIA_5', 'Q001_ord', 'Q002_ord', 'Q006_ord', 'Q022_ord', 'Q024_ord', 'Q025_ord']
CPU times: user 14.8 ms, sys: 781 μs, total: 15.6 ms
Wall time: 19.8 ms


In [28]:
# Primeiras linhas
df_base.head()

,NU_NOTA_MT,NU_NOTA_LC,NU_NOTA_CH,NU_NOTA_CN,NU_NOTA_REDACAO,TP_ESCOLA,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_PROVA,Q001,Q002,Q006,Q022,Q024,Q025,IN_TREINEIRO,CO_MUNICIPIO_PROVA_str,UF_CODE_PROVA,SG_UF_PROVA,REGIAO_ID_PROVA,REGIAO_NOME_PROVA,NOTA_MEDIA_5,Q001_ord,Q002_ord,Q006_ord,Q022_ord,Q024_ord,Q025_ord
0,335.2,424.8,375.6,476.0,0.0,2,2704302,Maceió,A,B,B,C,A,B,0,2704302,27,AL,2,Nordeste,322.32,1,2,2,3,1,2
1,635.0,571.1,600.2,584.4,660.0,2,3509502,Campinas,H,E,E,D,C,B,0,3509502,35,SP,3,Sudeste,610.14,8,5,5,4,3,2
2,528.5,504.8,536.5,432.8,560.0,2,3304557,Rio de Janeiro,A,E,B,B,A,B,0,3304557,33,RJ,3,Sudeste,512.52,1,5,2,2,1,2
3,603.3,522.0,536.6,533.5,560.0,1,2927408,Salvador,E,E,C,D,A,B,0,2927408,29,BA,2,Nordeste,551.08,5,5,3,4,1,2
4,396.9,489.9,548.0,398.8,520.0,2,2302602,Camocim,E,E,F,E,B,B,0,2302602,23,CE,2,Nordeste,470.72,5,5,6,5,2,2


In [29]:
# Informações sobre o dataset
df_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   NU_NOTA_MT              1000 non-null   float64
 1   NU_NOTA_LC              1000 non-null   float64
 2   NU_NOTA_CH              1000 non-null   float64
 3   NU_NOTA_CN              1000 non-null   float64
 4   NU_NOTA_REDACAO         1000 non-null   float64
 5   TP_ESCOLA               1000 non-null   int64  
 6   CO_MUNICIPIO_PROVA      1000 non-null   int64  
 7   NO_MUNICIPIO_PROVA      1000 non-null   object 
 8   Q001                    1000 non-null   object 
 9   Q002                    1000 non-null   object 
 10  Q006                    1000 non-null   object 
 11  Q022                    1000 non-null   object 
 12  Q024                    1000 non-null   object 
 13  Q025                    1000 non-null   object 
 14  IN_TREINEIRO            1000 non-null   i

---

## 3⃣ Extrair Variáveis de Trabalho do Dataset Original

Agora vamos ler apenas as colunas Q007 e Q008 do dataset original:

In [30]:
%%time
# Colunas a serem extraídas
colunas_trabalho = ['NU_INSCRICAO', 'Q007', 'Q008']

# Ler apenas as colunas necessárias
df_trabalho = pd.read_csv(
    RAW_FILE,
    sep=';',
    encoding='latin1',
    usecols=colunas_trabalho,
    low_memory=False
)

print(f"\n Variáveis de trabalho extraídas:")
print(f"  Linhas: {len(df_trabalho):,}")
print(f"  Colunas: {list(df_trabalho.columns)}")


 Variáveis de trabalho extraídas:
  Linhas: 3,933,955
  Colunas: ['NU_INSCRICAO', 'Q007', 'Q008']
CPU times: user 15.8 s, sys: 14.1 s, total: 29.9 s
Wall time: 41 s


In [31]:
# Primeiras linhas
df_trabalho.head(10)

,NU_INSCRICAO,Q007,Q008
0,210059085136,C,C
1,210059527735,A,B
2,210061103945,A,B
3,210060214087,A,B
4,210059980948,A,B
5,210058061539,A,B
6,210059855122,A,B
7,210058387333,A,B
8,210059085137,A,B
9,210060801601,A,B


In [32]:
# Verificar valores únicos
print(" Q007 - Situação de trabalho:")
print(df_trabalho['Q007'].value_counts(dropna=False).sort_index())

print("\n Q008 - Carga horária:")
print(df_trabalho['Q008'].value_counts(dropna=False).sort_index())

 Q007 - Situação de trabalho:
Q007
A    3611179
B     178524
C      39882
D     104370
Name: count, dtype: int64

 Q008 - Carga horária:
Q008
A      37878
B    2647503
C     855720
D     253087
E     139767
Name: count, dtype: int64


---

## 4⃣ Integrar Dados

Vamos juntar as variáveis de trabalho com o dataset base:

In [33]:
# Verificar se NU_INSCRICAO existe no dataset base
if 'NU_INSCRICAO' in df_base.columns:
    print(" NU_INSCRICAO encontrado no dataset base")
    coluna_join = 'NU_INSCRICAO'
else:
    # Se não existir, precisamos ler do arquivo original
    print(" NU_INSCRICAO não encontrado. Lendo índice do arquivo original...")
    df_inscricao = pd.read_csv(
        RAW_FILE,
        sep=';',
        encoding='latin1',
        usecols=['NU_INSCRICAO'],
        low_memory=False
    )
    df_base['NU_INSCRICAO'] = df_inscricao['NU_INSCRICAO']
    coluna_join = 'NU_INSCRICAO'
    print(" NU_INSCRICAO adicionado ao dataset base")

 NU_INSCRICAO não encontrado. Lendo índice do arquivo original...
 NU_INSCRICAO adicionado ao dataset base
 NU_INSCRICAO adicionado ao dataset base


In [34]:
# Realizar merge
df = df_base.merge(df_trabalho, on=coluna_join, how='left')

print(f"\n Merge concluído:")
print(f"  Linhas: {len(df):,}")
print(f"  Colunas: {len(df.columns)}")
print(f"\n Novas colunas adicionadas: Q007, Q008")


 Merge concluído:
  Linhas: 1,000
  Colunas: 30

 Novas colunas adicionadas: Q007, Q008


In [35]:
# Verificar dados ausentes
print(" Valores ausentes:")
print(df[['Q007', 'Q008']].isnull().sum())
print(f"\nTaxa de preenchimento Q007: {(1 - df['Q007'].isnull().mean()) * 100:.2f}%")
print(f"Taxa de preenchimento Q008: {(1 - df['Q008'].isnull().mean()) * 100:.2f}%")

 Valores ausentes:
Q007    0
Q008    0
dtype: int64

Taxa de preenchimento Q007: 100.00%
Taxa de preenchimento Q008: 100.00%


---

## 5⃣ Tratamento de Dados

### 5.1 Análise de Valores Ausentes

In [36]:
# Verificar se valores ausentes são aleatórios ou sistemáticos
print(" Análise de valores ausentes em Q007:")
print(f"\nTotal de ausentes: {df['Q007'].isnull().sum():,} ({df['Q007'].isnull().mean()*100:.2f}%)")

# Comparar com outras variáveis
print("\n Correlação de ausência com outras variáveis:")
if 'Q006' in df.columns:  # Renda
    print(f"  Q006 (Renda) também ausente: {(df['Q007'].isnull() & df['Q006'].isnull()).sum():,}")
if 'TP_ESCOLA' in df.columns:
    print(f"  Distribuição por escola:")
    print(df.groupby('TP_ESCOLA')['Q007'].apply(lambda x: x.isnull().mean() * 100))

 Análise de valores ausentes em Q007:

Total de ausentes: 0 (0.00%)

 Correlação de ausência com outras variáveis:
  Q006 (Renda) também ausente: 0
  Distribuição por escola:
TP_ESCOLA
1    0.0
2    0.0
3    0.0
Name: Q007, dtype: float64


### 5.2 Criar Labels Descritivos

In [37]:
# Mapeamento Q007 - Situação de trabalho
Q007_LABELS = {
    'A': 'Não trabalho',
    'B': 'Trabalho eventualmente',
    'C': 'Trabalho meio período',
    'D': 'Trabalho período integral'
}

# Mapeamento Q008 - Carga horária
Q008_LABELS = {
    'A': 'Nenhuma',
    'B': 'Até 10 horas',
    'C': '11 a 20 horas',
    'D': '21 a 30 horas',
    'E': '31 a 40 horas',
    'F': 'Mais de 40 horas'
}

# Aplicar mapeamentos
df['Q007_label'] = df['Q007'].map(Q007_LABELS)
df['Q008_label'] = df['Q008'].map(Q008_LABELS)

print(" Labels descritivos criados")
print("\n Q007_label:")
print(df['Q007_label'].value_counts(dropna=False))
print("\n Q008_label:")
print(df['Q008_label'].value_counts(dropna=False))

 Labels descritivos criados

 Q007_label:
Q007_label
Não trabalho                 964
Trabalho eventualmente        20
Trabalho período integral     12
Trabalho meio período          4
Name: count, dtype: int64

 Q008_label:
Q008_label
Até 10 horas     787
11 a 20 horas    172
21 a 30 horas     28
Nenhuma           10
31 a 40 horas      3
Name: count, dtype: int64


### 5.3 Criar Variáveis Ordinais

In [38]:
# Codificação ordinal Q007 (0 = não trabalha, 3 = período integral)
Q007_ORDINAL = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
df['Q007_ord'] = df['Q007'].map(Q007_ORDINAL)

# Codificação ordinal Q008 (0 = nenhuma, 5 = mais de 40h)
Q008_ORDINAL = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5}
df['Q008_ord'] = df['Q008'].map(Q008_ORDINAL)

print(" Variáveis ordinais criadas")
print("\n Distribuição Q007_ord:")
print(df['Q007_ord'].value_counts(dropna=False).sort_index())
print("\n Distribuição Q008_ord:")
print(df['Q008_ord'].value_counts(dropna=False).sort_index())

 Variáveis ordinais criadas

 Distribuição Q007_ord:
Q007_ord
0    964
1     20
2      4
3     12
Name: count, dtype: int64

 Distribuição Q008_ord:
Q008_ord
0     10
1    787
2    172
3     28
4      3
Name: count, dtype: int64


### 5.4 Criar Variáveis Derivadas

In [39]:
# Variável binária: trabalha ou não
df['TRABALHA'] = (df['Q007'] != 'A').astype(int)

# Categorias simplificadas de trabalho
def categorizar_trabalho(row):
    if pd.isna(row['Q007']):
        return np.nan
    elif row['Q007'] == 'A':
        return 'Não trabalha'
    elif row['Q007'] in ['B', 'C']:
        return 'Trabalho parcial'
    else:  # D
        return 'Período integral'

df['CATEGORIA_TRABALHO'] = df.apply(categorizar_trabalho, axis=1)

# Carga horária numérica (ponto médio dos intervalos)
CARGA_HORARIA_NUM = {
    'A': 0,
    'B': 5,    # Até 10h -> média 5h
    'C': 15.5, # 11-20h -> média 15.5h
    'D': 25.5, # 21-30h -> média 25.5h
    'E': 35.5, # 31-40h -> média 35.5h
    'F': 45    # Mais de 40h -> estimativa 45h
}
df['CARGA_HORARIA_NUM'] = df['Q008'].map(CARGA_HORARIA_NUM)

print(" Variáveis derivadas criadas")
print("\n TRABALHA:")
print(df['TRABALHA'].value_counts())
print("\n CATEGORIA_TRABALHO:")
print(df['CATEGORIA_TRABALHO'].value_counts())
print("\n CARGA_HORARIA_NUM - Estatísticas:")
print(df['CARGA_HORARIA_NUM'].describe())

 Variáveis derivadas criadas

 TRABALHA:
TRABALHA
0    964
1     36
Name: count, dtype: int64

 CATEGORIA_TRABALHO:
CATEGORIA_TRABALHO
Não trabalha        964
Trabalho parcial     24
Período integral     12
Name: count, dtype: int64

 CARGA_HORARIA_NUM - Estatísticas:
count    1000.00
mean        7.42
std         5.29
min         0.00
25%         5.00
50%         5.00
75%         5.00
max        35.50
Name: CARGA_HORARIA_NUM, dtype: float64
count    1000.00
mean        7.42
std         5.29
min         0.00
25%         5.00
50%         5.00
75%         5.00
max        35.50
Name: CARGA_HORARIA_NUM, dtype: float64


---

## 6⃣ Validação de Consistência

Verificar se há inconsistências entre Q007 e Q008:

In [40]:
# Criar tabela cruzada
print(" Tabela Cruzada Q007 × Q008:")
tabela_cruzada = pd.crosstab(
    df['Q007_label'], 
    df['Q008_label'], 
    margins=True,
    dropna=False
)
print(tabela_cruzada)

# Identificar inconsistências
# Exemplo: Q007='A' (não trabalha) mas Q008 != 'A' (tem carga horária)
inconsistencias = df[(df['Q007'] == 'A') & (df['Q008'] != 'A') & df['Q008'].notna()]
print(f"\n Inconsistências encontradas: {len(inconsistencias):,}")
print(f"   ({len(inconsistencias)/len(df)*100:.2f}% do total)")

if len(inconsistencias) > 0:
    print("\nExemplos de inconsistências:")
    print(inconsistencias[['Q007', 'Q007_label', 'Q008', 'Q008_label']].head(10))

 Tabela Cruzada Q007 × Q008:
Q008_label                 11 a 20 horas  21 a 30 horas  31 a 40 horas  \
Q007_label                                                               
Não trabalho                         165             25              2   
Trabalho eventualmente                 3              1              1   
Trabalho meio período                  1              1              0   
Trabalho período integral              3              1              0   
All                                  172             28              3   

Q008_label                 Até 10 horas  Nenhuma   All  
Q007_label                                              
Não trabalho                        762       10   964  
Trabalho eventualmente               15        0    20  
Trabalho meio período                 2        0     4  
Trabalho período integral             8        0    12  
All                                 787       10  1000  

 Inconsistências encontradas: 954
   (95.40% do tota

In [41]:
# Tratar inconsistências (opcional)
# Se pessoa diz não trabalhar (Q007=A) mas tem carga horária, corrigir
print(" Corrigindo inconsistências...")

# Criar cópia das variáveis originais
df['Q007_original'] = df['Q007']
df['Q008_original'] = df['Q008']

# Regra: Se Q007=A mas Q008 tem valor diferente de A, 
# assumir que Q008 está correto e ajustar Q007
mask_inconsistente = (df['Q007'] == 'A') & (df['Q008'].notna()) & (df['Q008'] != 'A')
if mask_inconsistente.sum() > 0:
    # Mapear Q008 para Q007 aproximado
    def q008_para_q007(q008):
        if q008 in ['B', 'C']:  # Até 20h
            return 'C'  # Meio período
        elif q008 in ['D', 'E', 'F']:  # Mais de 20h
            return 'D'  # Período integral
        return 'A'
    
    df.loc[mask_inconsistente, 'Q007'] = df.loc[mask_inconsistente, 'Q008'].apply(q008_para_q007)
    print(f"   Corrigidos: {mask_inconsistente.sum():,} registros")
    
    # Atualizar variáveis derivadas
    df.loc[mask_inconsistente, 'Q007_label'] = df.loc[mask_inconsistente, 'Q007'].map(Q007_LABELS)
    df.loc[mask_inconsistente, 'Q007_ord'] = df.loc[mask_inconsistente, 'Q007'].map(Q007_ORDINAL)
    df.loc[mask_inconsistente, 'TRABALHA'] = 1
    df.loc[mask_inconsistente, 'CATEGORIA_TRABALHO'] = df.loc[mask_inconsistente].apply(categorizar_trabalho, axis=1)

print("\n Inconsistências tratadas")

 Corrigindo inconsistências...
   Corrigidos: 954 registros

 Inconsistências tratadas

 Inconsistências tratadas


---

## 7⃣ Estatísticas Finais

In [42]:
print(" ESTATÍSTICAS FINAIS DO DATASET")
print("=" * 60)

print(f"\n Tamanho do dataset:")
print(f"  Total de registros: {len(df):,}")
print(f"  Total de colunas: {len(df.columns)}")

print(f"\n Situação de trabalho (Q007):")
print(df['Q007_label'].value_counts())
print(f"\n  Percentuais:")
print(df['Q007_label'].value_counts(normalize=True) * 100)

print(f"\n Carga horária (Q008):")
print(df['Q008_label'].value_counts())
print(f"\n  Percentuais:")
print(df['Q008_label'].value_counts(normalize=True) * 100)

print(f"\n Trabalha ou não:")
print(df['TRABALHA'].value_counts())
print(f"\n  Percentual que trabalha: {df['TRABALHA'].mean() * 100:.2f}%")

print(f"\n Categoria de trabalho:")
print(df['CATEGORIA_TRABALHO'].value_counts())

 ESTATÍSTICAS FINAIS DO DATASET

 Tamanho do dataset:
  Total de registros: 1,000
  Total de colunas: 39

 Situação de trabalho (Q007):
Q007_label
Trabalho meio período        931
Trabalho período integral     39
Trabalho eventualmente        20
Não trabalho                  10
Name: count, dtype: int64

  Percentuais:
Q007_label
Trabalho meio período        93.1
Trabalho período integral     3.9
Trabalho eventualmente        2.0
Não trabalho                  1.0
Name: proportion, dtype: float64

 Carga horária (Q008):
Q008_label
Até 10 horas     787
11 a 20 horas    172
21 a 30 horas     28
Nenhuma           10
31 a 40 horas      3
Name: count, dtype: int64

  Percentuais:
Q008_label
Até 10 horas     78.7
11 a 20 horas    17.2
21 a 30 horas     2.8
Nenhuma           1.0
31 a 40 horas     0.3
Name: proportion, dtype: float64

 Trabalha ou não:
TRABALHA
1    990
0     10
Name: count, dtype: int64

  Percentual que trabalha: 99.00%

 Categoria de trabalho:
CATEGORIA_TRABALHO
Trabalho par

In [43]:
# Resumo das novas colunas criadas
print("\n COLUNAS CRIADAS NESTE NOTEBOOK:")
print("=" * 60)
novas_colunas = [
    'Q007', 'Q008',  # Originais
    'Q007_label', 'Q008_label',  # Labels
    'Q007_ord', 'Q008_ord',  # Ordinais
    'TRABALHA',  # Binária
    'CATEGORIA_TRABALHO',  # Categórica simplificada
    'CARGA_HORARIA_NUM',  # Numérica
    'Q007_original', 'Q008_original'  # Backup
]
for col in novas_colunas:
    if col in df.columns:
        print(f"   {col}")


 COLUNAS CRIADAS NESTE NOTEBOOK:
   Q007
   Q008
   Q007_label
   Q008_label
   Q007_ord
   Q008_ord
   TRABALHA
   CATEGORIA_TRABALHO
   CARGA_HORARIA_NUM
   Q007_original
   Q008_original


---

## 8⃣ Salvar Dataset Processado

In [44]:
# Garantir que o diretório existe
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Salvar em formato parquet (mais eficiente)
print(f" Salvando dataset processado...")
df.to_parquet(OUTPUT_FILE, index=False, compression='snappy')

# Verificar tamanho do arquivo
tamanho_mb = OUTPUT_FILE.stat().st_size / (1024 * 1024)
print(f"\n Dataset salvo com sucesso!")
print(f"   Arquivo: {OUTPUT_FILE}")
print(f"   Tamanho: {tamanho_mb:.2f} MB")
print(f"   Registros: {len(df):,}")
print(f"   Colunas: {len(df.columns)}")

 Salvando dataset processado...



 Dataset salvo com sucesso!
   Arquivo: /home/interas/faculdade/ciencia-dados/enem-data-exploration/data/processed/enem_2023_trabalho_estudantil.parquet
   Tamanho: 0.07 MB
   Registros: 1,000
   Colunas: 39


In [45]:
# Também salvar uma versão CSV para facilitar inspeção
CSV_FILE = PROCESSED_DIR / 'enem_2023_trabalho_estudantil_sample.csv'

# Salvar amostra (ou dataset completo se menor que 10.000)
n_sample = min(10000, len(df))
if n_sample < len(df):
    df_sample = df.sample(n=n_sample, random_state=42)
else:
    df_sample = df.copy()

df_sample.to_csv(CSV_FILE, index=False)

print(f"\nAmostra CSV salva: {CSV_FILE}")
print(f"   Registros: {len(df_sample):,}")


Amostra CSV salva: /home/interas/faculdade/ciencia-dados/enem-data-exploration/data/processed/enem_2023_trabalho_estudantil_sample.csv
   Registros: 1,000


---

## 9⃣ Sumário de Qualidade dos Dados

In [46]:
# Criar relatório de qualidade
print(" RELATÓRIO DE QUALIDADE DOS DADOS")
print("=" * 60)

colunas_trabalho = ['Q007', 'Q008', 'Q007_ord', 'Q008_ord', 'TRABALHA', 
                    'CATEGORIA_TRABALHO', 'CARGA_HORARIA_NUM']

for col in colunas_trabalho:
    if col in df.columns:
        total = len(df)
        nulos = df[col].isnull().sum()
        validos = total - nulos
        taxa_preenchimento = (validos / total) * 100
        
        print(f"\n{col}:")
        print(f"  Total: {total:,}")
        print(f"  Válidos: {validos:,} ({taxa_preenchimento:.2f}%)")
        print(f"  Nulos: {nulos:,} ({(nulos/total)*100:.2f}%)")
        
        if df[col].dtype in ['int64', 'float64']:
            print(f"  Média: {df[col].mean():.2f}")
            print(f"  Mediana: {df[col].median():.2f}")

 RELATÓRIO DE QUALIDADE DOS DADOS

Q007:
  Total: 1,000
  Válidos: 1,000 (100.00%)
  Nulos: 0 (0.00%)

Q008:
  Total: 1,000
  Válidos: 1,000 (100.00%)
  Nulos: 0 (0.00%)

Q007_ord:
  Total: 1,000
  Válidos: 1,000 (100.00%)
  Nulos: 0 (0.00%)
  Média: 2.00
  Mediana: 2.00

Q008_ord:
  Total: 1,000
  Válidos: 1,000 (100.00%)
  Nulos: 0 (0.00%)
  Média: 1.23
  Mediana: 1.00

TRABALHA:
  Total: 1,000
  Válidos: 1,000 (100.00%)
  Nulos: 0 (0.00%)
  Média: 0.99
  Mediana: 1.00

CATEGORIA_TRABALHO:
  Total: 1,000
  Válidos: 1,000 (100.00%)
  Nulos: 0 (0.00%)

CARGA_HORARIA_NUM:
  Total: 1,000
  Válidos: 1,000 (100.00%)
  Nulos: 0 (0.00%)
  Média: 7.42
  Mediana: 5.00


---

##  Checklist de Conclusão

- [x] Dataset original carregado
- [x] Variáveis Q007 e Q008 extraídas
- [x] Merge realizado com sucesso
- [x] Labels descritivos criados
- [x] Variáveis ordinais criadas
- [x] Variáveis derivadas criadas (TRABALHA, CATEGORIA_TRABALHO, CARGA_HORARIA_NUM)
- [x] Inconsistências identificadas e tratadas
- [x] Dataset salvo em formato Parquet
- [x] Amostra CSV gerada para inspeção
- [x] Relatório de qualidade gerado

---

##  Próximos Passos

 **`03_analise_descritiva_trabalho.ipynb`**

No próximo notebook, iremos:
1. Analisar a distribuição de estudantes por situação de trabalho
2. Calcular estatísticas descritivas de desempenho por grupo
3. Criar visualizações comparativas
4. Traçar o perfil socioeconômico dos estudantes que trabalham

---

*Notebook criado em: 10 de dezembro de 2025*  
*Dataset processado: enem_2023_trabalho_estudantil.parquet*